# C6 Binary Action Classifier — SlowFast R50 (Fighting vs Normal)
Fine-tunes a Kinetics-400-pretrained SlowFast R50 for binary classification: **Fighting** vs **Normal**.

## Before running
1. Zip your two folders (`Fighting/` with 50 videos, `Normal/` with 195 videos) into one folder structure:
   ```
   dataset/
     Fighting/*.mp4 or .avi
     Normal/*.mp4 or .avi
   ```
2. Upload as a **private Kaggle Dataset** (Kaggle → Datasets → New Dataset → upload the zip, it auto-extracts).
3. Add it as input to this notebook (Notebook → Add Input → your dataset).
4. **Turn Internet ON** in notebook settings (right sidebar → Settings → Internet) — required to download pretrained SlowFast weights.
5. Set a GPU accelerator (P100 or T4x2) in notebook settings.
6. Edit `DATASET_ROOT` in the config cell below to match your dataset's actual mounted path (check the left sidebar under `/kaggle/input/` once added).

In [3]:
!pip install -q pytorchvideo decord av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.5 MB/s eta 0:00:00


In [4]:
import os, glob, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import decord
from decord import VideoReader
decord.bridge.set_bridge('native')
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
# ── CONFIG ──────────────────────────────────────────────────────────────
DATASET_ROOT = "/content/fighting_vs_normal"   # <-- EDIT THIS after adding your dataset
FIGHTING_DIR = os.path.join(DATASET_ROOT, "Fighting")
NORMAL_DIR   = os.path.join(DATASET_ROOT, "Normal")

NUM_FRAMES   = 32     # SlowFast R50 pretrained default clip length
SIZE         = 224
MAX_NORMAL_VIDEOS = 125   # subsample Normal down from 195 for rough class balance + faster training
BATCH_SIZE   = 4
EPOCHS       = 16
LR           = 1e-4
VAL_SPLIT    = 0.15
SEED         = 42
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [6]:
# ── BUILD MANIFEST (video-level, not clip-level) ──────────────────────────
def list_videos(folder):
    exts = (".mp4", ".avi", ".mov", ".mkv")
    return [f for f in glob.glob(os.path.join(folder, "*")) if f.lower().endswith(exts) and not os.path.basename(f).startswith("._")]

def is_readable(video_path, min_frames=2):
    """Quick validation: can decord actually open and read this file?"""
    try:
        vr = VideoReader(video_path)
        if len(vr) < min_frames:
            return False
        _ = vr.get_batch([0]).asnumpy()  # confirm at least one frame decodes
        return True
    except Exception:
        return False

def list_and_validate_videos(folder, label_name):
    candidates = list_videos(folder)
    valid, skipped = [], []
    for v in candidates:
        if is_readable(v):
            valid.append(v)
        else:
            skipped.append(v)
    print(f"[{label_name}] found {len(candidates)}, valid {len(valid)}, skipped (corrupt/unreadable) {len(skipped)}")
    if skipped:
        for s in skipped:
            print(f"    SKIPPED: {s}")
    return valid

fighting_videos = list_and_validate_videos(FIGHTING_DIR, "Fighting")
normal_videos_all = list_and_validate_videos(NORMAL_DIR, "Normal")

if len(normal_videos_all) > MAX_NORMAL_VIDEOS:
    normal_videos = random.sample(normal_videos_all, MAX_NORMAL_VIDEOS)
else:
    normal_videos = normal_videos_all

print(f"\nFinal counts -> Fighting: {len(fighting_videos)}, Normal (after cap): {len(normal_videos)}")

records = [(v, 1) for v in fighting_videos] + [(v, 0) for v in normal_videos]
random.shuffle(records)

n_val = max(1, int(len(records) * VAL_SPLIT))
val_records = records[:n_val]
train_records = records[n_val:]

print(f"Train videos: {len(train_records)}  Val videos: {len(val_records)}")
print(f"Train label balance -> Fighting: {sum(1 for _,l in train_records if l==1)}, Normal: {sum(1 for _,l in train_records if l==0)}")

[Fighting] found 0, valid 0, skipped (corrupt/unreadable) 0
[Normal] found 0, valid 0, skipped (corrupt/unreadable) 0

Final counts -> Fighting: 0, Normal (after cap): 0
Train videos: 0  Val videos: 0
Train label balance -> Fighting: 0, Normal: 0


In [7]:
# ── SHARED CLIP LOADER (same transform convention as project's clip_loader.py) ──
TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((SIZE, SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]),  # Kinetics norm, matches SlowFast pretraining
])

def load_clip(video_path, start_frame=0, num_frames=NUM_FRAMES, size=SIZE):
    """Decode num_frames consecutive frames starting at start_frame. Returns (T, C, H, W) tensor."""
    vr = VideoReader(video_path)
    total = len(vr)
    if total < num_frames:
        # video shorter than clip length: loop-pad by sampling with repetition
        idxs = np.linspace(0, total - 1, num_frames).astype(int)
    else:
        max_start = total - num_frames
        start_frame = min(start_frame, max_start)
        idxs = list(range(start_frame, start_frame + num_frames))
    frames = vr.get_batch(idxs).asnumpy()  # (T, H, W, C) RGB
    tensor_frames = torch.stack([TRANSFORM(f) for f in frames])  # (T, C, H, W)
    return tensor_frames

def random_start(video_path, num_frames=NUM_FRAMES):
    try:
        vr = VideoReader(video_path)
        total = len(vr)
    except Exception:
        return 0
    if total <= num_frames:
        return 0
    return random.randint(0, total - num_frames)

In [8]:
# ── PACK PATHWAY (SlowFast needs two input streams: slow + fast) ─────────
class PackPathway(nn.Module):
    def __init__(self, alpha=4):
        super().__init__()
        self.alpha = alpha
    def forward(self, frames_cthw):
        # frames_cthw: (C, T, H, W)
        fast_pathway = frames_cthw
        slow_idx = torch.linspace(0, frames_cthw.shape[1] - 1, frames_cthw.shape[1] // self.alpha).long()
        slow_pathway = torch.index_select(frames_cthw, 1, slow_idx)
        return [slow_pathway, fast_pathway]

pack_pathway = PackPathway(alpha=4)

class BinaryActionDataset(Dataset):
    def __init__(self, records, max_retries=5):
        self.records = records
        self.max_retries = max_retries

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        for attempt in range(self.max_retries):
            video_path, label = self.records[idx]
            try:
                start = random_start(video_path)
                clip = load_clip(video_path, start_frame=start)
                clip = clip.permute(1, 0, 2, 3)
                slow, fast = pack_pathway(clip)
                return slow, fast, torch.tensor(label, dtype=torch.float32)
            except Exception as e:
                print(f"[WARN] Failed to load {video_path} (attempt {attempt+1}/{self.max_retries}): {e}")
                idx = random.randint(0, len(self.records) - 1)
        raise RuntimeError(f"Exceeded max_retries ({self.max_retries}) trying to load a valid clip.")

def collate_fn(batch):
    slows, fasts, labels = zip(*batch)
    return torch.stack(slows), torch.stack(fasts), torch.stack(labels)

train_ds = BinaryActionDataset(train_records)
val_ds   = BinaryActionDataset(val_records)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# ── MODEL: SlowFast R50, pretrained on Kinetics-400, binary head ─────────
model = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=True)

in_features = model.blocks[-1].proj.in_features
model.blocks[-1].proj = nn.Linear(in_features, 1)   # binary logit output

model = model.to(device)
print(f"Replaced final projection layer: {in_features} -> 1 (binary)")

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_acc = 0.0
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "slowfast_fighting_binary.pth")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    t0 = time.time()
    for slow, fast, labels in train_loader:
        slow, fast, labels = slow.to(device), fast.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model([slow, fast]).squeeze(-1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    train_loss = total_loss / total
    train_acc = correct / total

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for slow, fast, labels in val_loader:
            slow, fast, labels = slow.to(device), fast.to(device), labels.to(device)
            logits = model([slow, fast]).squeeze(-1)
            loss = criterion(logits, labels)
            v_loss += loss.item() * labels.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            v_correct += (preds == labels).sum().item()
            v_total += labels.size(0)
    val_loss = v_loss / max(v_total, 1)
    val_acc = v_correct / max(v_total, 1)

    elapsed = time.time() - t0
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss {train_loss:.4f} acc {train_acc:.3f} | val_loss {val_loss:.4f} acc {val_acc:.3f} | {elapsed:.1f}s")

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch,
            "num_frames": NUM_FRAMES,
            "alpha": 4,
            "size": SIZE,
        }, best_ckpt_path)
        print(f"  -> saved new best checkpoint (val_acc={val_acc:.3f})")

print(f"\nTraining complete. Best val_acc: {best_val_acc:.3f}")
print(f"Checkpoint saved to: {best_ckpt_path}")

## Download the checkpoint
Once training finishes, go to the **Output** pane on the right side of the Kaggle notebook, find `checkpoints/slowfast_fighting_binary.pth`, and download it.
This file is everything you need to bring back into the project — see the integration steps in the main response.